In [1]:
from google.colab import drive
import os
import glob
import re
import pandas as pd
from tqdm.notebook import tqdm

# Монтируем Google Drive
drive.mount('/content/drive')

# Путь к папке с субъектами OASIS-1. Все папки с субьектами находятся в одной папке
OASIS_ROOT = '/content/drive/MyDrive/mri/oasis_processed_all_subjects'

Mounted at /content/drive


In [ ]:
# Проверим наличие папок с субьектами
!ls -h "/content/drive/MyDrive/mri/oasis_processed_all_subjects" | head -5

OAS1_0001_MR1
OAS1_0002_MR1
OAS1_0003_MR1
OAS1_0004_MR1
OAS1_0005_MR1


In [ ]:
!ls -h "/content/drive/MyDrive/mri/oasis_processed_all_subjects" | tail -5

OAS1_0453_MR1
OAS1_0454_MR1
OAS1_0455_MR1
OAS1_0456_MR1
OAS1_0457_MR1


Нам необходимо преобразовать отчеты FreeSurfer, которые находятся в папке stats для каждого субьекта (aseg.stats, lh.aparc.stats, rh.aparc.stats) для всех субъектов из набора OASIS-1 в единую плоскую таблицу (DataFrame), где каждая строка это один субъект, а каждый столбец это конкретная анатомическая метрика, для этого можно использовать  парсер файлов на Python и не устанавливать тяжелое окружение FreeSurfer и не запускать asegstats2table и aparcstats2table внутри Google Colab.

Устройство файлов *.stats в FreeSurfer

Каждый *.stats файл состоит из трех частей:
1. Заголовки с глобальными метриками: строки вида # Measure. В них содержатся важнейшие общие показатели мозга.
2. Описание столбцов таблицы: строка вида # ColHeaders, задающая имена колонок для основной таблицы.
3. Основная таблица данных: строки без символа # в начале, разделенные пробелами. Каждая строка — отдельная структура.

In [2]:
# Подготовим универсальную функцию, которая читает любой *.stats файл, извлекает
# как глобальные метрики из # Measure, так и табличные значения для
# каждой анатомической структуры,
# и разворачивает их в плоский словарь {имя_признака: значение}.

# Замечание: Датасет OASIS-1 обрабатывался в 2007 году,
# когда утилита выгружала в таблицу абсолютно все строки из файла
# разметки aparc.annot.ctab, включая нулевую техническую метку unknown.
def parse_freesurfer_stats(filepath, prefix="", target_metrics=None):
    """
    Парсит файл FreeSurfer .stats и возвращает плоский словарь метрик.

    :param filepath: путь к файлу (например, .../stats/aseg.stats)
    :param prefix: префикс для имен столбцов ('aseg', 'lh_aparc', 'rh_aparc')
    :param target_metrics: список нужных столбцов из таблицы
    """
    data = {}
    col_headers = []
    # *Эти две строки не являются корой головного мозга
    IGNORE_STRUCTS = {'unknown', 'corpuscallosum'}

    if not os.path.exists(filepath):
        return data

    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # 1. Извлекаем глобальные метрики из строк "# Measure"
        if line.startswith('# Measure'):
            # Пример: # Measure BrainSeg, BrainSegVol, Brain Segmentation Volume, 1123456.0, mm^3
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 4:
                short_name = parts[1]  # Например, BrainSegVol или eTIV
                try:
                    val = float(parts[3])
                    col_name = f"{prefix}_{short_name}" if prefix else short_name
                    data[col_name] = val
                except ValueError:
                    pass

        # 2. Извлекаем названия столбцов таблицы из "# ColHeaders"
        elif line.startswith('# ColHeaders'):
            col_headers = line.replace('# ColHeaders', '').strip().split()

        # 3. Парсим строки основной таблицы (без '#')
        elif not line.startswith('#') and col_headers:
            values = line.split()
            if len(values) != len(col_headers):
                continue

            row_dict = dict(zip(col_headers, values))
            struct_name = row_dict.get('StructName')
            if not struct_name or struct_name.lower() in IGNORE_STRUCTS:
                continue

            # Очищаем имя структуры от лишних символов
            for col_name, val in row_dict.items():
                if col_name in ('Index', 'SegId', 'StructName'):
                    continue
                if target_metrics and col_name not in target_metrics:
                    continue
                try:
                    num_val = float(val)
                    # Формируем плоское имя: например, lh_aparc_superiortemporal_ThickAvg
                    flat_key = f"{prefix}_{struct_name}_{col_name}" if prefix else f"{struct_name}_{col_name}"
                    data[flat_key] = num_val
                except ValueError:
                    pass

    return data

In [3]:
# Задаем, какие метрики из таблиц мы хотим вытащить
ASEG_METRICS = ['Volume_mm3', 'normMean']
APARC_METRICS = ['SurfArea', 'GrayVol', 'ThickAvg', 'ThickStd', 'MeanCurv']

all_subjects_data = []

# Ищем все папки субъектов (предполагаем, что внутри каждой есть папка stats)
subject_dirs = sorted([
    d for d in glob.glob(os.path.join(OASIS_ROOT, '*'))
    if os.path.isdir(d) and os.path.isdir(os.path.join(d, 'stats'))
])

print(f"Найдено папок субъектов со статистикой: {len(subject_dirs)}")

for subj_dir in tqdm(subject_dirs, desc="Парсинг субъектов"):
    subject_id = os.path.basename(subj_dir)
    stats_dir = os.path.join(subj_dir, 'stats')

    subj_record = {'subject_id': subject_id}

    # 1. Парсим подкорковую сегментацию (aseg.stats)
    aseg_path = os.path.join(stats_dir, 'aseg.stats')
    subj_record.update(parse_freesurfer_stats(
        aseg_path, prefix='aseg', target_metrics=ASEG_METRICS
    ))

    # 2. Парсим кору левого полушария (lh.aparc.stats)
    lh_aparc_path = os.path.join(stats_dir, 'lh.aparc.stats')
    subj_record.update(parse_freesurfer_stats(
        lh_aparc_path, prefix='lh', target_metrics=APARC_METRICS
    ))

    # 3. Парсим кору правого полушария (rh.aparc.stats)
    rh_aparc_path = os.path.join(stats_dir, 'rh.aparc.stats')
    subj_record.update(parse_freesurfer_stats(
        rh_aparc_path, prefix='rh', target_metrics=APARC_METRICS
    ))

    all_subjects_data.append(subj_record)

# Создаем итоговый DataFrame
df_oasis = pd.DataFrame(all_subjects_data)
df_oasis.set_index('subject_id', inplace=True)

print(f"Итоговый размер таблицы: {df_oasis.shape[0]} субъектов, {df_oasis.shape[1]} признаков")

Найдено папок субъектов со статистикой: 425


Парсинг субъектов:   0%|          | 0/425 [00:00<?, ?it/s]

Итоговый размер таблицы: 425 субъектов, 435 признаков


In [4]:
print(f"Общая размерность df_oasis: {df_oasis.shape}")
print("-" * 45)
print(f"Признаков из aseg.stats:     {len([c for c in df_oasis.columns if c.startswith('aseg_')])}")
print(f"Признаков из lh.aparc.stats: {len([c for c in df_oasis.columns if c.startswith('lh_')])}")
print(f"Признаков из rh.aparc.stats: {len([c for c in df_oasis.columns if c.startswith('rh_')])}")
print(f"Пропущенных значений (NaN):  {df_oasis.isna().sum().sum()}")

Общая размерность df_oasis: (425, 435)
---------------------------------------------
Признаков из aseg.stats:     101
Признаков из lh.aparc.stats: 167
Признаков из rh.aparc.stats: 167
Пропущенных значений (NaN):  0


In [5]:
# Посмотрим все колонки aseg, которые НЕ заканчиваются на Volume_mm3 или normMean (то есть глобальные # Measure)
global_aseg = [c for c in df_oasis.columns if c.startswith('aseg_') and not c.endswith(('Volume_mm3', 'normMean'))]
print("Глобальные метрики aseg:", global_aseg)

Глобальные метрики aseg: ['aseg_BrainMaskNVox', 'aseg_BrainMaskVol', 'aseg_BrainSegNVox', 'aseg_BrainSegVol', 'aseg_ICV']


In [8]:
# Ищем колонки aseg с нулевой дисперсией (константы или все нули)
aseg_cols = [c for c in df_oasis.columns if c.startswith('aseg_')]
zero_std_cols = df_oasis[aseg_cols].std()[lambda x: x == 0].index.tolist()

print(f"Колонки aseg с нулевым разбросом ({len(zero_std_cols)} шт.):")
print(zero_std_cols)

Колонки aseg с нулевым разбросом (6 шт.):
['aseg_Left-choroid-plexus_Volume_mm3', 'aseg_Left-choroid-plexus_normMean', 'aseg_WM-hypointensities_Volume_mm3', 'aseg_WM-hypointensities_normMean', 'aseg_non-WM-hypointensities_Volume_mm3', 'aseg_non-WM-hypointensities_normMean']


In [9]:
# Удаляем константные (нулевые) колонки и дублирующие счетчики вокселей
cols_to_drop = zero_std_cols + ['aseg_BrainMaskNVox', 'aseg_BrainSegNVox']
df_oasis.drop(columns=[c for c in cols_to_drop if c in df_oasis.columns], inplace=True)

print(f"Размерность после очистки технических колонок aseg: {df_oasis.shape}")

Размерность после очистки технических колонок aseg: (425, 427)


In [10]:
display(df_oasis.head())

,aseg_BrainMaskVol,aseg_BrainSegVol,aseg_ICV,aseg_Left-Cerebral-White-Matter_Volume_mm3,aseg_Left-Cerebral-White-Matter_normMean,aseg_Left-Cerebral-Cortex_Volume_mm3,aseg_Left-Cerebral-Cortex_normMean,aseg_Left-Lateral-Ventricle_Volume_mm3,aseg_Left-Lateral-Ventricle_normMean,aseg_Left-Inf-Lat-Vent_Volume_mm3,...,rh_temporalpole_SurfArea,rh_temporalpole_GrayVol,rh_temporalpole_ThickAvg,rh_temporalpole_ThickStd,rh_temporalpole_MeanCurv,rh_transversetemporal_SurfArea,rh_transversetemporal_GrayVol,rh_transversetemporal_ThickAvg,rh_transversetemporal_ThickStd,rh_transversetemporal_MeanCurv
subject_id,,,,,,,,,,,,,,,,,,,,,
OAS1_0001_MR1,1285238.0,921495.0,1.347498e+06,156115.0,108.2804,188486.0,71.4874,27330.0,21.9581,1487.0,...,430.0,2553.0,3.490,0.847,0.162,267.0,676.0,2.153,0.456,0.121
OAS1_0002_MR1,1231611.0,878237.0,1.151269e+06,166466.0,107.4070,184712.0,70.3166,3482.0,36.0548,243.0,...,390.0,2099.0,3.640,1.076,0.156,295.0,777.0,2.341,0.361,0.108
OAS1_0003_MR1,1395001.0,950261.0,1.403415e+06,172606.0,107.4490,198592.0,72.1212,22239.0,22.2445,1877.0,...,445.0,2245.0,2.904,0.759,0.170,310.0,686.0,2.005,0.515,0.123
OAS1_0004_MR1,1607482.0,1162451.0,1.593285e+06,220306.0,107.0284,240933.0,66.0211,16017.0,23.3043,627.0,...,512.0,2309.0,3.238,0.801,0.154,282.0,767.0,2.349,0.407,0.132
OAS1_0005_MR1,1797642.0,1368245.0,1.733779e+06,248654.0,107.6196,309574.0,67.8282,4966.0,34.9422,647.0,...,631.0,3159.0,3.504,0.726,0.155,313.0,824.0,2.363,0.403,0.110


In [11]:
# Проверяем наличие пропущенных значений
missing_counts = df_oasis.isna().sum()
if missing_counts.sum() > 0:
    print("Внимание! Найдены пропуски в данных:")
    print(missing_counts[missing_counts > 0].head(10))
else:
    print("Пропусков нет, все файлы успешно обработаны!")

Пропусков нет, все файлы успешно обработаны!


In [12]:
# Сохраняем результат на Google Drive в формате csv
output_csv = os.path.join(OASIS_ROOT, 'oasis_freesurfer_flat.csv')
df_oasis.to_csv(output_csv)
print(f"Таблица сохранена в: {output_csv}")

Таблица сохранена в: /content/drive/MyDrive/mri/oasis_processed_all_subjects/oasis_freesurfer_flat.csv


In [13]:
!head /content/drive/MyDrive/mri/oasis_processed_all_subjects/oasis_freesurfer_flat.csv

subject_id,aseg_BrainMaskVol,aseg_BrainSegVol,aseg_ICV,aseg_Left-Cerebral-White-Matter_Volume_mm3,aseg_Left-Cerebral-White-Matter_normMean,aseg_Left-Cerebral-Cortex_Volume_mm3,aseg_Left-Cerebral-Cortex_normMean,aseg_Left-Lateral-Ventricle_Volume_mm3,aseg_Left-Lateral-Ventricle_normMean,aseg_Left-Inf-Lat-Vent_Volume_mm3,aseg_Left-Inf-Lat-Vent_normMean,aseg_Left-Cerebellum-White-Matter_Volume_mm3,aseg_Left-Cerebellum-White-Matter_normMean,aseg_Left-Cerebellum-Cortex_Volume_mm3,aseg_Left-Cerebellum-Cortex_normMean,aseg_Left-Thalamus-Proper_Volume_mm3,aseg_Left-Thalamus-Proper_normMean,aseg_Left-Caudate_Volume_mm3,aseg_Left-Caudate_normMean,aseg_Left-Putamen_Volume_mm3,aseg_Left-Putamen_normMean,aseg_Left-Pallidum_Volume_mm3,aseg_Left-Pallidum_normMean,aseg_3rd-Ventricle_Volume_mm3,aseg_3rd-Ventricle_normMean,aseg_4th-Ventricle_Volume_mm3,aseg_4th-Ventricle_normMean,aseg_Brain-Stem_Volume_mm3,aseg_Brain-Stem_normMean,aseg_Left-Hippocampus_Volume_mm3,aseg_Left-Hippocampus_normMean,aseg_Left

In [14]:
# Сохраняем результат на Google Drive в формате parquet
output_parquet = os.path.join(OASIS_ROOT, 'oasis_freesurfer_flat.parquet')
df_oasis.to_parquet(output_parquet, engine='pyarrow', compression='snappy')
print(f"Файл сохранен по пути: {output_parquet}")

Файл сохранен по пути: /content/drive/MyDrive/mri/oasis_processed_all_subjects/oasis_freesurfer_flat.parquet
